In [1]:
import json
from pathlib import Path
import pickle
from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display


project_root = Path(Path.cwd()).resolve().parent

labels_dir = project_root / 'data' / 'potential_labels'
model_path = project_root / 'models' / 'embeddings.pkl'

In [2]:
# Load a small pre-trained embedding model (open local model)
model = SentenceTransformer('all-MiniLM-L6-v2')  # open-source embedding model

# Ensure the labels directory exists
if not labels_dir.exists():
    print(f"Directory {labels_dir} does not exist. Please create it and add JSON files.")
    exit(1)

# Collect JSON files
files = sorted([p for p in labels_dir.iterdir() if p.is_file() and p.suffix == '.json'])

progress = widgets.IntProgress(value=0, min=0, max=len(files), description='Progress:')
pct = widgets.HTML(value="0%")
display(widgets.HBox([progress, pct]))
embedding_db = []  # list to store {"id", "name", "vector"} for each recipe
for idx, fp in enumerate(files):
    try:
        text = fp.read_text(encoding='utf-8')
        obj = json.loads(text)   
    except Exception as e:
        # Log and continue on parse/read errors
        print(f'Skipping {fp.name}: {e}')
        continue

    progress.value = idx + 1
    pct.value = f"{int(100 * (idx + 1) / len(files))}%"

    # add source filename so downstream steps know which file produced this record
    if isinstance(obj, dict) and obj.get("title") and obj.get("ingredients"):
        title_instructions_file = {
            
        }
        text = f"{obj["title"]}: " + ", ".join(obj["ingredients"])
        vector = model.encode(text)            # get embedding vector (e.g., numpy array)
        embedding_db.append({
            "title" : obj["title"],
            "ingredients": obj["ingredients"],
            "source_file": fp.name,
            "vector": vector
        })
        
progress.value = len(files)
pct.value = "100%"

pickle.dump(embedding_db, open(model_path, 'wb'))
print(f"Stored {len(embedding_db)} recipe embeddings in the database.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 19945 recipe embeddings in the database.


In [2]:
embedding_db = pickle.load(open(model_path, 'rb'))

In [ ]:
import numpy as np

def find_best_matches(query_ingredients, top_n: int = 10):
    """
    Return the top_n matching recipe entries from embedding_db for the given
    query_ingredients list. Returns a list of tuples: (global_index, entry_dict, score).
    """
    if not embedding_db:
        return []

    # Embed the query ingredients list into a vector
    query_text = ", ".join(query_ingredients)
    q_vec = np.array(model.encode(query_text))

    # Stack all stored vectors into a matrix (N x D)
    mat = np.vstack([np.array(entry["vector"]) for entry in embedding_db])

    # Compute cosine similarities in a vectorized way
    q_norm = np.linalg.norm(q_vec)
    mat_norms = np.linalg.norm(mat, axis=1)
    denom = mat_norms * (q_norm if q_norm != 0 else 1e-12)

    # avoid division by zero
    denom[denom == 0] = 1e-12
    sims = (mat @ q_vec) / denom

    # Get top_n indices (highest similarity)
    k = min(int(top_n), sims.size)
    top_idx = np.argsort(sims)[::-1][:k]

    # Build results: (global_index, entry, score)
    return [(embedding_db[i], float(sims[i])) for i in top_idx]
    


# Example query
user_ingredients = ["cheese", "bread", "mustard", "pickle"]  # user has these ingredients
matches = find_best_matches(user_ingredients)
for entry, score in matches:
    print(f"{entry['title']} [{entry['source_file']}] — {score * 100:.2f}%")

Fried Pickles [recipe_02953.json] — 69.63%
Hamburger with Double Cheddar Cheese, Grilled Vidalia Onion and Horseradish Mustard [recipe_02949.json] — 68.83%
Dill Pickle Bacon Grilled Cheese [recipe_09286.json] — 67.79%
Fried Bread and Butter Pickles with Buttermilk Chive Dressing [recipe_02865.json] — 67.64%
Hunter's sauce [recipe_19391.json] — 66.46%
Ultimate Cheeseburger Loaf [recipe_15679.json] — 65.51%
Carnitas Bowl [recipe_11546.json] — 65.45%
Breakfast Casserole [recipe_03546.json] — 65.32%
Muffuletta-Style Grilled Ham-and-Cheese Sandwiches [recipe_03876.json] — 65.16%
Coffee-Rubbed Cheeseburgers with Texas Barbecue Sauce [recipe_02822.json] — 65.14%


In [21]:
import requests
import os

useOpenAI = True

user_ingredients = ["carrots", "edamame", "corn", "pork"]  # user has these ingredients

# Suppose we take the top 3 matches from the embedding search (for demonstration)
# Here we'll use the embedding search result; in practice, you would sort and pick top N.
top_matches = find_best_matches(user_ingredients, top_n=25)

# Construct a prompt with the query and top matches for GPT-4.1-nano
candidate_list_str = ""
for i, (recipe, score) in enumerate(top_matches, start=1):
    ingr_list = ", ".join(recipe["ingredients"])
    candidate_list_str += f'{{ "recipe_name": "{recipe['title']}", "file_name": "{recipe['source_file']}", "ingredients":  "{ingr_list}" }}'

recipe_style = "recipe that is good for lunch"

system_msg = (
    "You are a helpful cooking assistant. A user has certain ingredients, and we have some candidate recipes from a database. "
    "Choose which recipe is the best match for the user's ingredients and give a reason for choosing it."
)
user_msg = (
    f"The user has the following ingredients: {', '.join(user_ingredients)}.\n"
    f"The candidate recipes are:\n[{candidate_list_str}]\n"
    f"The user wants a recipe that matches ``{recipe_style}``\n"
    "Which recipe from the candidate recipes best matches the user's ingredients and has the style that user wants?" 
    "Respond with the recipe name and file_name and the reason for picking this recipe in JSON. Like this: "
    "{\"recipe_name\": \"Tomato Soup\", \"file_name\": \"recipe_00031.json\", \"reason\": \"Because soup is good food\"}"
)

if (useOpenAI and os.getenv("OPENAI_API_KEY")):
    API_KEY = os.getenv("OPENAI_API_KEY") # load from env
    api_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    model_name = "gpt-4.1-nano" # Cheapest fast model currently from OpenAI (5.1-nano takes 5 times longer)
else:
    # Use Ollama local model server
    api_url = "http://127.0.0.1:11434/v1/chat/completions"
    headers = {
        "Content-Type": "application/json"
    }
    model_name = "gemma3:1b"

data = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
}

if (not model_name.startswith("gpt-5") ):
    data["temperature"] = 0.3

# Send the HTTP request to OpenAI
response = requests.post(api_url, headers=headers, json=data)
response_json = response.json()

if (response_json.get("error") or response.status_code != 200):
    print(f"Error: {response.status_code} - {response.text}")
else:
    # Extract the assistant's answer (recipe title and id)
    best_recipe_answer = response_json["choices"][0]["message"]["content"].strip()
    print(f"{model_name}'s answer: {best_recipe_answer}")

gpt-4.1-nano's answer: {"recipe_name": "Saucy Pork Lettuce Wraps", "file_name": "recipe_12803.json", "reason": "This recipe uses pork and is designed as a quick, light lunch with the option to include vegetables like carrots and shredded cabbage, matching the user's ingredients and lunch preference."}
